<a href="https://colab.research.google.com/github/MaineCalabrezi13/DiagnosticoPlantas/blob/main/Diagnostico_de_Plantas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [65]:
!pip install streamlit
!pip install pillow
!pip install pyngrok
!pip install ultralytics
!pip install kaggle

In [66]:
%%writefile app.py

import streamlit as st
from ultralytics import YOLO
from PIL import Image
import os

# ============================================
# CONFIGURAÇÕES DA PÁGINA
# ============================================

st.set_page_config(
    page_title="AgroScan AI",
    page_icon="🌱",
    layout="centered"
)

# ============================================
# CARREGAR MODELO TREINADO
# ============================================

@st.cache_resource
def carregar_modelo():
    return YOLO("best.pt")

try:
    modelo = carregar_modelo()

except Exception as erro:

    st.error(
        f"Erro ao carregar o modelo: {erro}"
    )

    st.stop()

# ============================================
# DICIONÁRIO DE EXPLICAÇÕES
# ============================================

RECOMENDACOES = {

    # --- PIMENTÃO ---
    "Pepper__bell___Bacterial_spot": {
        "diagnostico": "Pimentão — Mancha bacteriana",
        "explicacao": "Foram identificadas lesões foliares úmidas e escuras características de infecção bacteriana.",
        "acao": "Eliminar restos culturais contaminados, evitar irrigação por aspersão excessiva e utilizar produtos à base de cobre se recomendado."
    },

    "Pepper__bell___healthy": {
        "diagnostico": "Pimentão — Saudável",
        "explicacao": "A folha do pimentão apresenta coloração e aspecto saudáveis, sem sinais detectáveis de fitopatologias.",
        "acao": "Manter o manejo nutricional e monitoramento periódico preventivo."
    },

    # --- BATATA ---
    "Potato___Early_blight": {
        "diagnostico": "Batata — Pinta preta / Alternariose",
        "explicacao": "Foram detectadas manchas circulares de cor marrom ou preta com anéis concêntricos nas folhas mais velhas.",
        "acao": "Aplicar fungicidas protetores ou sistêmicos específicos, evitar o plantio adensado e monitorar a umidade foliar."
    },

    "Potato___Late_blight": {
        "diagnostico": "Batata — Requeima",
        "explicacao": "Identificados sintomas severos de queima foliar com áreas necróticas úmidas, indicando alta agressividade do patógeno.",
        "acao": "Realizar controle químico fitossanitário imediato com fungicidas apropriados e remover focos iniciais da doença."
    },

    "Potato___healthy": {
        "diagnostico": "Batata — Saudável",
        "explicacao": "A folhagem da batateira apresenta-se em ótimas condições de vigor, sem sintomas aparentes.",
        "acao": "Continuar o monitoramento sistemático e as boas práticas de manejo fitotécnico."
    },

    # --- TOMATE ---
    "Tomato_Bacterial_spot": {
        "diagnostico": "Tomate — Mancha bacteriana",
        "explicacao": "Pequenas manchas escuras e angulares circundadas por halos amarelados foram encontradas na folha.",
        "acao": "Evitar o trabalho nos campos com plantas molhadas e pulverizar defensivos recomendados à base de cobre."
    },

    "Tomato_Early_blight": {
        "diagnostico": "Tomate — Pinta preta / Alternariose",
        "explicacao": "Constatada a presença de lesões foliares com pontuações necróticas concêntricas típicas de alternariose.",
        "acao": "Promover a rotação de culturas, podar as folhas baixeiras afetadas e aplicar fungicidas adequados."
    },

    "Tomato_Late_blight": {
        "diagnostico": "Tomate — Requeima",
        "explicacao": "Foram identificadas manchas grandes e irregulares de aspecto esverdeado-escuro ou oleoso nas folhas.",
        "acao": "Pulverizar fungicidas sistêmicos preventivos/curativos e otimizar o arejamento da cultura."
    },

    "Tomato_Leaf_Mold": {
        "diagnostico": "Tomate — Bolor cinzento / Mofo das folhas",
        "explicacao": "Presença de uma cobertura aveludada cinza a esverdeada na face inferior da folha, associada a manchas amareladas na face superior.",
        "acao": "Reduzir a umidade relativa dentro da estufa/lavoura e melhorar o espaçamento entre plantas."
    },

    "Tomato_Septoria_leaf_spot": {
        "diagnostico": "Tomate — Septoriose",
        "explicacao": "Foram detectadas numerosas manchas pequenas, circulares, com centros cinzentos e bordas escuras.",
        "acao": "Eliminar restos de plantas após a colheita e aplicar o controle químico apropriado."
    },

    "Tomato_Spider_mites_Two-spotted_spider_mite": {
        "diagnostico": "Tomate — Ácaro-rajado",
        "explicacao": "Sinais de bronzeamento, pequenas pontuações claras nas folhas e presença de teias finas causadas por ácaros.",
        "acao": "Aplicar acaricidas específicos de forma localizada ou introduzir predadores naturais."
    },

    "Tomato__Target_Spot": {
        "diagnostico": "Tomate — Mancha-alvo",
        "explicacao": "Identificadas lesões circulares planas com círculos concêntricos bem definidos.",
        "acao": "Garantir uma boa drenagem do solo e aplicar fungicidas."
    },

    "Tomato__Tomato_Yellow_Leaf_Curl_Virus": {
        "diagnostico": "Tomate — Vírus do enrolamento amarelo",
        "explicacao": "Folhas superiores exibem redução de tamanho e enrolamento.",
        "acao": "Controlar a mosca-branca."
    },

    "Tomato__Tomato_mosaic_virus": {
        "diagnostico": "Tomate — Vírus do mosaico",
        "explicacao": "Folhas apresentam padrão de mosaico.",
        "acao": "Remover plantas contaminadas."
    },

    "Tomato_healthy": {
        "diagnostico": "Tomate — Saudável",
        "explicacao": "Folha sem anomalias aparentes.",
        "acao": "Manter o plano atual."
    }

}

# ============================================
# TÍTULO
# ============================================

st.title(
    "🌱 AgroScan AI – Diagnóstico Automatizado de Fitopatologias"
)

st.write(
"""
Faça o upload de uma imagem para que o sistema realize
uma análise automática baseada em inteligência artificial.
"""
)

st.divider()

# ============================================
# UPLOAD DA IMAGEM
# ============================================

imagem = st.file_uploader(
    "Selecione uma imagem",
    type=[
        "jpg",
        "jpeg",
        "png",
        "jfif"
    ]
)

# ============================================
# PROCESSAMENTO
# ============================================

if imagem:

    img = Image.open(imagem)

    col1, col2 = st.columns(2)

# ============================================
# COLUNA DA IMAGEM
# ============================================

    with col1:

        st.subheader(
            "Imagem enviada"
        )

        st.image(
            img,
            use_container_width=True
        )

        # ============================================
        # INFORMAÇÕES DA IMAGEM
        # ============================================

        st.subheader(
            "Informações da imagem"
        )

        st.write(
            f"📄 Nome: {imagem.name}"
        )

        st.write(
            f"🖼️ Formato: {img.format}"
        )

        st.write(
            f"📐 Resolução: {img.width} x {img.height}"
        )

# ============================================
# IA PROCESSANDO
# ============================================

    with st.spinner(
        "Analisando imagem..."
    ):

        caminho_temp="imagem_temp.jpg"

        if img.mode in ("RGBA","P"):

            img_para_salvar=img.convert(
                "RGB"
            )

        else:

            img_para_salvar=img

        img_para_salvar.save(
            caminho_temp,
            "JPEG"
        )

        resultados=modelo(
            caminho_temp,
            verbose=False
        )

        os.remove(
            caminho_temp
        )

        resultado_ia=resultados[0]

        id_classe=resultado_ia.probs.top1

        nome_classe=resultado_ia.names[
            id_classe
        ]

        confianca=float(
            resultado_ia.probs.top1conf
        )*100

# ============================================
# COLUNA RESULTADOS
# ============================================

    with col2:

        st.subheader(
            "Resultado"
        )

        if confianca<50:

            st.error(
                "Não foi possível identificar a doença com segurança."
            )

            st.write(
                "A confiança da IA ficou abaixo do limite mínimo estabelecido."
            )

            st.warning(
                "Sugestão: envie uma imagem mais nítida."
            )

        else:

            if nome_classe in RECOMENDACOES:

                info=RECOMENDACOES[
                    nome_classe
                ]

                st.success(
                    info["diagnostico"]
                )

                st.subheader(
                    "Explicação"
                )

                st.write(
                    info["explicacao"]
                )

                st.subheader(
                    "Recomendação"
                )

                st.warning(
                    info["acao"]
                )

# ============================================
# CONFIANÇA
# ============================================

        st.subheader(
            "Grau de confiança"
        )

        st.progress(
            int(confianca)
        )

        st.write(
            f"{confianca:.2f}%"
        )

st.divider()

# ============================================
# LIMITAÇÕES
# ============================================

with st.expander(
    "Limitações do sistema"
):

    st.write(
"""
• O modelo foi treinado apenas com folhas presentes no dataset.

• Imagens desfocadas podem reduzir a precisão.

• Objetos que não sejam folhas podem gerar classificações incorretas.

• O sistema serve como apoio à decisão e não substitui análise especializada.
"""
)

st.caption(
    "AgroScan AI – Diagnóstico Automatizado de Fitopatologias"
)

Overwriting app.py


In [67]:
!streamlit run app.py &>/content/logs.txt &

In [68]:
from pyngrok import ngrok

ngrok.set_auth_token(
"3EYNDwCYVwNjTd7AYR46VJ6geDT_3HVQjSSJX72v8wnu9efc8"
)

In [69]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://slideshow-litter-outspoken.ngrok-free.dev" -> "http://localhost:8501"
